In [4]:
import os
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from ultralytics import YOLO
from IPython.display import display, clear_output
import ipywidgets as widgets
import io

## 0️⃣ Setup & Configuration

In [6]:
warnings.filterwarnings('ignore')

# ── CONFIG ──────────────────────────────────────────────────────────────────
DATA_DIR     = r"C:\Users\ASUS G 16\Desktop\fish classify"
TEST_DIR     = os.path.join(DATA_DIR, "test")
PLOTS_DIR    = os.path.join(DATA_DIR, "plots")
WEIGHTS_PATH = os.path.join(DATA_DIR, "yolo_fish_augmented", "weights", "best.pt")
os.makedirs(PLOTS_DIR, exist_ok=True)

print("✅ Setup complete")
print(f" Data directory : {DATA_DIR}")
print(f" Weights path   : {WEIGHTS_PATH}")

✅ Setup complete
 Data directory : C:\Users\ASUS G 16\Desktop\fish classify
 Weights path   : C:\Users\ASUS G 16\Desktop\fish classify\yolo_fish_augmented\weights\best.pt


## 1️⃣ Train YOLOv8 Model

In [7]:
model = YOLO("yolov8n-cls.pt")  # nano classification, pretrained on ImageNet

results = model.train(
    data=DATA_DIR,
    epochs=30,
    imgsz=224,
    batch=16,
    project=DATA_DIR,
    name="yolo_fish_augmented",

    # ── On-the-fly Augmentation ──────────────────────────────────────────────
    # Note: for YOLOv8 classification mode, only color & flip augmentations
    # are applied. Geometric ones (degrees/translate/scale/shear) are ignored.
    # Offline augmentation for weak classes was already done in notebook 01.
    fliplr=0.5,    # horizontal flip (50% probability)
    flipud=0.0,    # no vertical flip
    hsv_h=0.015,   # hue variation
    hsv_s=0.50,    # saturation variation
    hsv_v=0.40,    # brightness variation
    erasing=0.25,  # random erasing

    patience=10,       # early stopping patience
    pretrained=True,   # use ImageNet pretrained weights
    exist_ok=True      # allow overwriting existing run folder
)

print("\n✅ YOLOv8 training complete!")
print(f"📁 Best weights: {WEIGHTS_PATH}")

New https://pypi.org/project/ultralytics/8.4.72 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.27  Python-3.13.7 torch-2.12.0+cpu CPU (13th Gen Intel Core i7-13650HX)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\ASUS G 16\Desktop\fish classify, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.25, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, 

## 2️⃣ Training Results — Accuracy & Loss Curves

In [8]:
results_csv = os.path.join(DATA_DIR, "yolo_fish_augmented", "results.csv")
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

epochs   = df["epoch"]
top1_acc = df["metrics/accuracy_top1"]
top5_acc = df["metrics/accuracy_top5"]
t_loss   = df["train/loss"]
v_loss   = df["val/loss"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("YOLOv8 Training Results", fontsize=14, fontweight="bold")

axes[0].plot(epochs, top1_acc, label="Top-1 Accuracy", color="steelblue")
axes[0].plot(epochs, top5_acc, label="Top-5 Accuracy", color="orange")
axes[0].set_title("YOLOv8 Accuracy over Epochs")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, t_loss, label="Train Loss",      color="steelblue")
axes[1].plot(epochs, v_loss, label="Validation Loss", color="orange")
axes[1].set_title("YOLOv8 Loss over Epochs")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "yolo_training_results.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Training results saved")
print(f"   Best Top-1 accuracy: {top1_acc.max()*100:.2f}%")

<Figure size 1400x500 with 2 Axes>

✅ Training results saved
   Best Top-1 accuracy: 99.20%


## 3️⃣ Evaluate on Test Set

In [9]:
model = YOLO(WEIGHTS_PATH)
class_names_yolo = model.names
name_to_idx = {v: k for k, v in class_names_yolo.items()}

y_true, y_pred = [], []
total = sum(
    len(os.listdir(os.path.join(TEST_DIR, c)))
    for c in os.listdir(TEST_DIR)
    if os.path.isdir(os.path.join(TEST_DIR, c))
)
done = 0

for class_folder in sorted(os.listdir(TEST_DIR)):
    class_path = os.path.join(TEST_DIR, class_folder)
    if not os.path.isdir(class_path) or class_folder not in name_to_idx:
        continue
    true_idx = name_to_idx[class_folder]
    for img_file in os.listdir(class_path):
        img_path = os.path.join(class_path, img_file)
        try:
            result   = model(img_path, verbose=False)[0]
            pred_idx = int(result.probs.top1)
            y_true.append(true_idx)
            y_pred.append(pred_idx)
        except:
            pass
        done += 1
        if done % 100 == 0:
            print(f"  Progress: {done}/{total}", end="\r")

accuracy  = accuracy_score(y_true, y_pred) * 100
precision = precision_score(y_true, y_pred, average="weighted", zero_division=0) * 100
recall    = recall_score(y_true, y_pred, average="weighted", zero_division=0) * 100
f1        = f1_score(y_true, y_pred, average="weighted", zero_division=0) * 100

print("\n" + "=" * 45)
print("  YOLOv8 Fish Classification Results")
print("  Evaluated on: TEST SET")
print("=" * 45)
print(f"  Accuracy : {accuracy:.2f} %")
print(f"  Precision: {precision:.2f} %")
print(f"  Recall   : {recall:.2f} %")
print(f"  F1-Score : {f1:.2f} %")
print("=" * 45)

yolo_metrics = {
    "Accuracy" : round(accuracy, 2),
    "Precision": round(precision, 2),
    "Recall"   : round(recall, 2),
    "F1-Score" : round(f1, 2)
}
with open(os.path.join(DATA_DIR, "yolo_metrics.json"), "w") as f:
    json.dump(yolo_metrics, f)
print("✅ Metrics saved to yolo_metrics.json")

WARNING ss: 1700/1761
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs


  YOLOv8 Fish Classification Results
  Evaluated on: TEST SET
  Accuracy : 98.47 %
  Precision: 98.50 %
  Recall   : 98.47 %
  F1-Score : 98.46 %
✅ Metrics saved to yolo_metrics.json


## 4️⃣ Sample Predictions (3×3 Grid)

In [10]:
random.seed(42)
all_classes    = [c for c in sorted(os.listdir(TEST_DIR))
                  if os.path.isdir(os.path.join(TEST_DIR, c))]
sample_classes = random.sample(all_classes, min(9, len(all_classes)))

sample_images, sample_labels = [], []
for class_name in sample_classes:
    class_path = os.path.join(TEST_DIR, class_name)
    files = [f for f in os.listdir(class_path)
             if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    if files:
        sample_images.append(os.path.join(class_path, random.choice(files)))
        sample_labels.append(class_name)

fig, axes = plt.subplots(3, 3, figsize=(14, 14))
fig.suptitle("Sample Predictions — YOLOv8", fontsize=16, fontweight="bold")

for i, ax in enumerate(axes.flatten()):
    if i >= len(sample_images):
        ax.axis("off"); continue
    img_path   = sample_images[i]
    true_label = sample_labels[i]
    img        = Image.open(img_path).convert("RGB")
    result     = model(img_path, verbose=False)[0]
    pred_idx   = int(result.probs.top1)
    pred_label = class_names_yolo[pred_idx]
    confidence = float(result.probs.top1conf) * 100
    color      = "green" if pred_label == true_label else "red"
    ax.imshow(img); ax.axis("off")
    ax.set_title(
        f"Pred: {pred_label}\nTrue: {true_label}\nConf: {confidence:.2f}%",
        fontsize=9, color=color, pad=6
    )

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "yolo_sample_predictions.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Sample predictions saved")

<Figure size 1400x1400 with 9 Axes>

✅ Sample predictions saved


## 5️⃣ Interactive Demo — Upload & Predict

In [11]:
CONFIDENCE_THRESHOLD = 70

upload = widgets.FileUpload(accept="image/*", multiple=False)
output = widgets.Output()
display(upload, output)

def predict_image(change):
    with output:
        clear_output(wait=True)
        if not upload.value:
            return
        try:
            uploaded_file = (list(upload.value.values())[0]
                             if isinstance(upload.value, dict) else upload.value[0])
            image_bytes  = uploaded_file["content"]
            original_img = Image.open(io.BytesIO(image_bytes)).convert("RGB")

            r         = model(original_img, verbose=False)[0]
            top3_idx  = r.probs.top5[:3]
            top3_conf = r.probs.top5conf[:3]
            predictions = [(r.names[int(i)], float(c) * 100)
                           for i, c in zip(top3_idx, top3_conf)]
            fish_name, confidence = predictions[0]

            img   = original_img.copy()
            scale = 900 / img.width
            img   = img.resize((900, int(img.height * scale)), Image.LANCZOS)
            draw  = ImageDraw.Draw(img)
            try:
                font = ImageFont.truetype("arial.ttf", 32)
            except:
                font = ImageFont.load_default()

            if confidence >= CONFIDENCE_THRESHOLD:
                text = f"Fish Type: {fish_name}\nConfidence: {confidence:.2f}%"
            else:
                lines = ["I'm not sure, but possibly:"]
                for name, conf in predictions:
                    lines.append(f"{name} — {conf:.1f}%")
                text = "\n".join(lines)

            bbox = draw.multiline_textbbox((0, 0), text, font=font, spacing=10)
            tw, th = bbox[2]-bbox[0], bbox[3]-bbox[1]
            pad = 22
            x1, y1 = img.width - tw - 2*pad - 20, 20
            x2, y2 = img.width - 20, y1 + th + 2*pad
            draw.rectangle([x1, y1, x2, y2], fill="black")
            draw.multiline_text((x1+pad, y1+pad), text,
                                fill="white", font=font, spacing=10)

            if confidence >= CONFIDENCE_THRESHOLD:
                print(f"Fish Type : {fish_name}")
                print(f"Confidence: {confidence:.2f}%")
            else:
                print("I'm not sure, but possibly:")
                for name, conf in predictions:
                    print(f"  {name}: {conf:.2f}%")
            display(img)
        except Exception as e:
            print("Something went wrong:", e)

upload.observe(predict_image, names="value")

FileUpload(value=(), accept='image/*', description='Upload')

Output()

# Camera vision

In [ ]:
import cv2

In [ ]:
import cv2
import numpy as np
from PIL import Image
from ultralytics import YOLO

CONFIDENCE_THRESHOLD = 70
model_cam = YOLO(WEIGHTS_PATH)

print("🎥 Starting camera... Press Q to quit")

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("❌ Camera not found. Make sure your webcam is connected.")
else:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Convert frame to PIL for YOLO
        img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        # Predict
        result     = model_cam(img_pil, verbose=False)[0]
        top3_idx   = result.probs.top5[:3]
        top3_conf  = result.probs.top5conf[:3]
        predictions = [(result.names[int(i)], float(c) * 100)
                       for i, c in zip(top3_idx, top3_conf)]
        fish_name, confidence = predictions[0]

        # Draw overlay box on frame
        h, w = frame.shape[:2]
        overlay = frame.copy()

        if confidence >= CONFIDENCE_THRESHOLD:
            lines = [
                f"Fish: {fish_name}",
                f"Confidence: {confidence:.1f}%"
            ]
            box_color = (0, 200, 0)   # green
        else:
            lines = [
                "Not sure, possibly:",
                f"1. {predictions[0][0]} ({predictions[0][1]:.1f}%)",
                f"2. {predictions[1][0]} ({predictions[1][1]:.1f}%)",
                f"3. {predictions[2][0]} ({predictions[2][1]:.1f}%)",
            ]
            box_color = (0, 165, 255)  # orange

        # Background box
        font       = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.65
        thickness  = 2
        line_h     = 30
        padding    = 12
        box_w      = 380
        box_h      = padding * 2 + line_h * len(lines)
        x1, y1     = 10, 10
        x2, y2     = x1 + box_w, y1 + box_h

        cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
        cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)

        for j, line in enumerate(lines):
            cv2.putText(frame, line,
                        (x1 + padding, y1 + padding + line_h * (j + 1) - 8),
                        font, font_scale, (255, 255, 255), thickness)

        # Instructions
        cv2.putText(frame, "Press Q to quit",
                    (w - 180, h - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1)

        cv2.imshow("YOLOv8 Fish Classifier - Camera", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    print(" Camera closed")